In [ ]:
import comtradeapicall as cd
import pandas as pd
import numpy as np
from comtrade import Comtrade
import requests
import os
from datetime import datetime
import matplotlib.pyplot as plt
from io import BytesIO
from dotenv import load_dotenv

load_dotenv()

print(os.getenv("COMTRADE_API_KEY"))

d4457e079e6240cc8f74def00b4e192f


In [2]:
url_list= {'Reporter_codes': 'https://comtradeapi.un.org/files/v1/app/reference/Reporters.json', 
           'Partner_codes': 'https://comtradeapi.un.org/files/v1/app/reference/partnerAreas.json'} # https://docs.ropensci.org/comtradr/reference/country_codes.html
# these links go to json files that contain all country codes in UN Comtrade database

codes_dict = {}

try:          #this loop is needed to 2 concise dataframes with country codes
    for name, link in url_list.items():
        data = requests.get(link)
        data.raise_for_status()
        data = data.json()
        
        df = pd.DataFrame(data['results'])
        df.rename(columns = {'text': 'country'}, inplace=True)
        df.set_index('country', inplace=True)
        
        codes_dict[name] = df

except Exception as e:
    print('An error occured, ', e)

In [3]:
def get_code(country: str, df=codes_dict['Partner_codes']) -> int:
    """
    This function returns a code of country you need to proceed with API-call.
    Input: name of country (needs to be the same as in database), output: country's ISO code.
    By default, locates country in Reporter codes. Works only with Partner_codes and Reporter_codes dataframes.
    """
    try:
        return df.loc[country, 'PartnerCode']
    except Exception as e:
        return df.loc[country, 'reporterCode']

def get_country(code: int, df=codes_dict['Partner_codes']) -> str:
    """
    This function returns name of country from ISO code you provide.
    Input: ISO code of country, output: name of country.
    By default, locates country in Partner codes. Works only with Partner_codes and Reporter_codes dataframes.
    """
    dfn = df.reset_index()
    try:
        dfn.set_index('PartnerCode', inplace=True)
    except Exception:
        dfn.set_index('reporterCode', inplace=True)
    return dfn.loc[code, 'country']

In [4]:
nat_gas_codes = ['271111', # Natural gas, liquefied
                 '271121' # Natural gas, gaseous
                 ]
ng_codes_str = ",".join(nat_gas_codes) # api call requires comma-separated string, not list

year_list = list(range(2000, datetime.now().year)) # list of all years from 2000 to now

def month_adder(year: int) -> str:
    """
    This function creates a string of YYYYMM values to call api on every month in this year.
    Input: one year you need to create year+month from.
    Output: string of 12 months of this year.
    """
    year_list = [str(year) + '0' + str(x) if x < 10 else str(year) + str(x) for x in range(1, 13)]
    return ','.join(year_list)


In [5]:
folder_name = 'nat_gas_2000-2025_Trade_data'
os.makedirs(folder_name, exist_ok=True)

In [6]:
# WARNING: TO DOWNLOAD THE DATA YOU NEED ABOUT 60-70 MINUTES!!!

for year in year_list:      
    path = os.path.join(folder_name, str(year))

    if os.path.exists(path):
        print(f'The data for {year} already exists. Proceeding with next year.')
        continue

    print(f'Getting data for {year}')
    try:
        data = cd.getFinalData(subscription_key =os.getenv("COMTRADE_API_KEY"), typeCode='C', 
                           freqCode='M', clCode='HS', period=month_adder(year), reporterCode=None,
                           cmdCode=ng_codes_str, flowCode=None, partnerCode=None, partner2Code=None, 
                           customsCode=None, motCode=None, format_output = 'JSON')
        print(f'Data for {year} is downloaded')
    except Exception as e:
        print(f'An error occured for {year}', e)
    
    data.to_csv(path, index=False)


The data for 2000 already exists. Proceeding with next year.
The data for 2001 already exists. Proceeding with next year.
The data for 2002 already exists. Proceeding with next year.
The data for 2003 already exists. Proceeding with next year.
The data for 2004 already exists. Proceeding with next year.
The data for 2005 already exists. Proceeding with next year.
The data for 2006 already exists. Proceeding with next year.
The data for 2007 already exists. Proceeding with next year.
The data for 2008 already exists. Proceeding with next year.
The data for 2009 already exists. Proceeding with next year.
The data for 2010 already exists. Proceeding with next year.
The data for 2011 already exists. Proceeding with next year.
The data for 2012 already exists. Proceeding with next year.
The data for 2013 already exists. Proceeding with next year.
The data for 2014 already exists. Proceeding with next year.
The data for 2015 already exists. Proceeding with next year.
The data for 2016 alread

In [ ]:
df_raw = pd.concat(
    [pd.read_csv(os.path.join(folder_name, str(year))) for year in year_list], ignore_index=True
)
df_raw

In [ ]:
df_raw.sample(5).style

In [ ]:
plt.figure(figsize=(10,5))
df_raw.isna().sum().sort_values().plot.bar()

In [ ]:
cols_to_keep = ['period', 'refYear', 'refMonth', 'reporterCode', 
                'flowCode', 'partnerCode', 'classificationCode',
                'cmdCode', 'customsCode', 'motCode', 'qtyUnitCode', 'qty', 
                'isQtyEstimated', 'altQtyUnitCode', 'altQty', 'isAltQtyEstimated',
                'netWgt', 'isNetWgtEstimated', 'primaryValue', 'isAggregate']

explanations = {'period': 'YYYYMM of trade', 
                'refYear': 'year of trade', 
                'refMonth': 'month of trade', 
                'reporterCode': 'code of country who reported the trade', 
                'flowCode': 'code of trade flow, usually "M" for "Import" and "X" for "Export" ', 
                'partnerCode': 'code of country who is a counterparty in the trade reported by reporter country ', 
                'classificationCode': 'type of Harmonized System (HS) classification used for this trade. It has changed throughout the years',
                'cmdCode': 'code of commodity traded', 
                'customsCode': 'defines how the goods entered or left the economic territory', 
                'motCode': 'code of transport used to deliver the trade', 
                'qtyUnitCode': 'defines the unit of measurement of commodity traded', 
                'qty': 'quantity of commodity traded measured in a way specified in qtyUnitCode', 
                'isQtyEstimated': 'Is quantity Estimated - boolean', 
                'altQtyUnitCode': 'alternative unit of measurement of commodity traded', 
                'altQty': 'quantity of commodity traded measured in altQtyUnitCode units', 
                'isAltQtyEstimated': 'Is alternative quantity Estimated - boolean',
                'netWgt': 'net weight of commodity traded in kilograms', 
                'isNetWgtEstimated': 'Is quantity Estimated - boolean', 
                'primaryValue': 'value of trade in dollars',
                'isAggregate': 'is value aggregated - True or False'}


In [ ]:
df = df_raw[cols_to_keep].copy()
df

In [ ]:
response = requests.get('https://comtradeapi.un.org/files/v1/app/wiki/ComtradePlus_DataItems.xlsx')
response.raise_for_status()
if response.status_code == 200:
    excel_codes = pd.ExcelFile(BytesIO(response.content))
print(excel_codes.sheet_names)       

#It is difficult to make everything nicer here (e.g by function) so I needed to create manually these dfs for further comfortable usage
# This dataframes are needed to convert codes in some columns to human-readable words.

flow_codes = pd.read_excel(excel_codes, sheet_name='REF FLOWS')
flow_codes.set_index('flwCode', inplace=True)

mot_codes = pd.read_excel(excel_codes, sheet_name = 'REF MOT')
mot_codes.set_index('motCode', inplace=True)

qty_codes = pd.read_excel(excel_codes, sheet_name = 'REF QTY')
qty_codes.set_index('qtyCode', inplace=True)

customs_codes = pd.read_excel(excel_codes, sheet_name = 'REF CUSTOMS')
customs_codes.set_index('cstCode', inplace=True)

country_codes = pd.read_excel(excel_codes, sheet_name = 'REF COUNTRIES')
country_codes.set_index('geoAreaCode', inplace=True)

In [ ]:
df['reporter'] = df.reporterCode.apply(lambda x: country_codes.loc[x]) # adding name of reporters
df['partner'] = df.partnerCode.apply(lambda x: country_codes.loc[x]) # adding name of partners

df['flow'] = df.flowCode.apply(lambda x: "Import" if "M" in x else "Export" if "X" in x else "None") # adding category of flow: Import/Export
df['flow_detail'] = df.flowCode.apply(lambda x: flow_codes.loc[x, 'flwDescription']) # adding detailed type of flow (Re-import etc.)

df['transport'] = df.motCode.apply(lambda x: mot_codes.loc[x]) # adding name of category of transport

df['qty_type'] = df.qtyUnitCode.apply(lambda x: qty_codes.loc[x, 'qtyAbbr']) # adding unit symbols of quantities (kg - kilograms, m^3. -cubic meters, NaN - unspecified )
df['alt_qty_type'] = df.altQtyUnitCode.apply(lambda x: qty_codes.loc[x, 'qtyDescription']) # here I added description indtead of unit symbols because units are very different and not understandable
df['customs_type'] = df.customsCode.apply(lambda x: customs_codes.loc[x]) #adding names of custom procedures


In [ ]:
df['period'] = pd.to_datetime(df['period'].astype(str), format = '%Y%m') # now we have it in date format to easily work with time series
df['primaryValue'] = pd.to_numeric(df['primaryValue'], errors="coerce") #everything to numeric
df['netWgt'] = pd.to_numeric(df['netWgt'], errors="coerce")

In [ ]:
aggr_keys = {'parter': 'World', 'customs': 'TOTAL CPC', 
             'transport': 'Total Modes of Transport'} #These keys can be used in future to filter some types of aggregation

In [ ]:
dfr = df[df.isAggregate == False].copy() #Now we have dataframe that contains only reports without any type of aggregation
dfr = dfr[(dfr.flow_detail == 'Export') | (dfr.flow_detail == 'Import')] 
# It appeared that sub-categories double actual values arificially, as for this df we want no aggregated data we filter this

dfr.drop('flow_detail', axis=1, inplace=True) #We have column "Flow" for this, no need to repeat yourself

"""
That was extremely difficult to find why the numbers does not match, but the reason is found - 
even if we no aggregation sub-categories like 'Foreign Import', 'Domestic Export' etc. can report the same trade 2 times.
Now we have clean like a tear, excellent dataframe to work with.
"""

In [ ]:
dfr.sample(7).style

In [ ]:
for i in dfr.columns:
    print(f'column {i} has {dfr[i].isna().sum()} n/a rows ({dfr[i].isna().sum()*100/dfr.shape[0]}%).') #How many NaN values each column has